Variable dependiente (target del modelo base)

Este cuaderno define la **variable objetivo** del modelo, `VarDep`: la etiqueta de
**bueno / malo** que el scoring aprenderá a predecir. Toma como entrada la base
consolidada del notebook `01` (`info.pkl`).

## La idea: punto de observación, historial y desempeño

Cada crédito se mira en un **punto de observación** (`FECHA_CORTE`). A partir de ese
instante el tiempo se divide en dos:

- **Historial de buró** — *hacia atrás*: el comportamiento del cliente antes de la
  observación. Son las **variables predictoras** (se construyeron en `01`).
- **Desempeño** — *hacia adelante*: cómo evoluciona el crédito en los **12 meses
  siguientes** (M2–M13). De aquí se deriva la **`VarDep`**.

```
        ◄─────  HISTORIAL DE BURÓ  ─────┃─────  DESEMPEÑO  ─────►
              (variables predictoras)   ┃   (define la VarDep)
                                        ┃
   ···   36M   24M   12M   6M   3M      ┃    M1   M2  M3 ··· M12  M13
         └────────────────────────┘     │    └───────────────┘
            ventanas hacia atrás        ┃        ventana 12 meses
            (creadas en notebook 01)    ┃          (M2–M13)
                                        ┃
                                 PUNTO DE OBSERVACIÓN  (FECHA_CORTE)
```

> las variables predictoras solo pueden venir del **pasado**; la
> etiqueta, del **futuro**. Por eso, al final, las columnas de desempeño se eliminan
> de la base de modelado usarlas como predictoras sería *fuga de datos*.

## Categorías de la `VarDep`

A partir del desempeño, cada crédito recibe una de **6 categorías**:

| VarDep | Categoría | Regla |
|---|---|---|
| **0** | Bueno | máx. mora en M2–M13 = 0 días |
| **1** | Malo | máx. mora en M2–M13 > 60 días |
| **2** | Indeterminado | máx. mora entre 1 y 60 días (zona gris) |
| **3** | Vencido | mora en M1 > 180 días (ya llega deteriorado) |
| **4** | Sin desempeño | < 6 meses con saldo en la institución |
| **5** | No bancarizado | sin score de buró (`score419`) |

El modelo se entrena **solo con 0 (bueno) y 1 (malo)**; las demás categorías son
exclusiones.

In [16]:
# %% Ejemplo ilustrativo: cómo se lee la VarDep en la ventana de desempeño
import pandas as pd

# Datos NO reales — solo para explicar el concepto.
# Valores = días de morosidad. El historial muestra la mora en cada ventana.
cols = pd.MultiIndex.from_tuples(
    [("Historial de buró  (predictoras, hacia atrás)", w)
     for w in ["36M", "24M", "12M", "6M", "3M"]]
    + [("Punto obs.", "M1")]
    + [("Ventana de desempeño  (M2–M13  →  define la VarDep)", f"M{i}")
       for i in range(2, 14)]
    + [("Resultado", "VarDep")]
)
ejemplo = pd.DataFrame(
    [
        [ 0,  0,  0, 0, 0,    0,   0, 0,  0,  0,   0,  0, 0, 0, 0, 0, 0, 0,  "0 · Bueno"],
        [30, 10,  0, 0, 0,    0,   0, 0, 45, 90, 120, 70, 0, 0, 0, 0, 0, 0,  "1 · Malo"],
        [ 0,  0,  0, 0, 0,    0,   0,15, 25, 10,   0,  0, 0, 0, 0, 0, 0, 0,  "2 · Indeterminado"],
        [60, 90, 30, 0, 0,  210,   0, 0,  0,  0,   0,  0, 0, 0, 0, 0, 0, 0,  "3 · Vencido >180"],
    ],
    index=["Cliente A", "Cliente B", "Cliente C", "Cliente D"],
    columns=cols,
)


def color_mora(v):
    if not isinstance(v, (int, float)):
        return ""                                  # celda de texto (VarDep)
    if v == 0:
        return "background-color:#c6efce"          # verde  — sin mora
    if v <= 30:
        return "background-color:#ffeb9c"          # amarillo
    if v <= 60:
        return "background-color:#ffcc99"          # naranja
    return "background-color:#ffc7ce;font-weight:bold"   # rojo — mora grave


(ejemplo.style
    .map(color_mora)
    .set_caption("Ejemplo ilustrativo — días de mora por mes.  "
                 "Verde = 0 · amarillo ≤ 30 · naranja ≤ 60 · rojo > 60")
    .set_table_styles([{"selector": "caption",
                        "props": "caption-side:bottom; font-size:0.9em; padding-top:6px"}]))

## 1. Cargar datos

In [17]:
# %% Cargar la base consolidada (salida del notebook 01)
import pandas as pd
import numpy as np

info = pd.read_pickle("data/buro/info.pkl")
print("Dimensiones:", info.shape)

Dimensiones: (79591, 2565)


In [18]:
# %% Población de créditos otorgados, por cohorte (FECHA_CORTE)
info["FECHA_CORTE"].value_counts().sort_index()

FECHA_CORTE
2021-09-30    14304
2021-12-31    14665
2022-03-31    15630
2022-06-30    16862
2022-09-30    18130
Name: count, dtype: int64

## 2. Desempeño en la institución

El **desempeño** es el comportamiento del crédito en la ventana de 12 meses (M2–M13).

- `DESEMPENO` = número de meses con **saldo de deuda > 0**: cuántos meses el cliente
  mantuvo el crédito activo en la institución.
- `SIN_DESEMPENO` = clientes con **menos de 6 meses** de saldo — no hay suficiente
  comportamiento que observar para etiquetarlos con confianza.

In [19]:
# %% DESEMPENO = nº de meses (M2..M13) con saldo de deuda > 0
cols_saldo = [f"SALDO_DEUDA_OP_M{i}" for i in range(2, 14)]   # M2..M13
info["DESEMPENO"] = (info[cols_saldo] > 0).sum(axis=1)

info.groupby("DESEMPENO").size().reset_index(name="N").sort_values("DESEMPENO")

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/2014525713.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["DESEMPENO"] = (info[cols_saldo] > 0).sum(axis=1)


,DESEMPENO,N
0,0,96
1,1,137
2,2,159
3,3,152
4,4,188
5,5,247
6,6,246
7,7,273
8,8,342
9,9,438


In [20]:
# %% SIN_DESEMPENO: < 6 meses con saldo
info["SIN_DESEMPENO"] = np.where(info["DESEMPENO"] < 6, "SIN_DESEMPENO", "CON_DESEMPENO")

info.groupby("SIN_DESEMPENO").size().reset_index(name="N")

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/1557159932.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["SIN_DESEMPENO"] = np.where(info["DESEMPENO"] < 6, "SIN_DESEMPENO", "CON_DESEMPENO")


,SIN_DESEMPENO,N
0,CON_DESEMPENO,78612
1,SIN_DESEMPENO,979


In [21]:
# %% SIN_DESEMPENO por cohorte
(info.groupby(["FECHA_CORTE", "SIN_DESEMPENO"]).size()
     .reset_index(name="N").sort_values("FECHA_CORTE"))

,FECHA_CORTE,SIN_DESEMPENO,N
0,2021-09-30,CON_DESEMPENO,14118
1,2021-09-30,SIN_DESEMPENO,186
2,2021-12-31,CON_DESEMPENO,14481
3,2021-12-31,SIN_DESEMPENO,184
4,2022-03-31,CON_DESEMPENO,15421
5,2022-03-31,SIN_DESEMPENO,209
6,2022-06-30,CON_DESEMPENO,16645
7,2022-06-30,SIN_DESEMPENO,217
8,2022-09-30,CON_DESEMPENO,17947
9,2022-09-30,SIN_DESEMPENO,183


## 3. Marca Bancarizado

`MARCA_BANCARIZADO` indica si el cliente tiene historial en el buró de crédito
(`score419`). Un cliente **no bancarizado** (sin score) no se puede modelar con
variables de buró.

In [22]:
# %% MARCA_BANCARIZADO: ¿tiene score de buró (score419)?
info["MARCA_BANCARIZADO"] = np.where(
    (info["score419"] == 0) | (info["score419"].isna()),
    "NO BANCARIZADO",
    "BANCARIZADO",
)

info.groupby("MARCA_BANCARIZADO").size().reset_index(name="N")

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/2916087009.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["MARCA_BANCARIZADO"] = np.where(


,MARCA_BANCARIZADO,N
0,BANCARIZADO,78859
1,NO BANCARIZADO,732


In [23]:
# %% MARCA_BANCARIZADO por cohorte
(info.groupby(["FECHA_CORTE", "MARCA_BANCARIZADO"]).size()
     .reset_index(name="N").sort_values("FECHA_CORTE"))

,FECHA_CORTE,MARCA_BANCARIZADO,N
0,2021-09-30,BANCARIZADO,14182
1,2021-09-30,NO BANCARIZADO,122
2,2021-12-31,BANCARIZADO,14570
3,2021-12-31,NO BANCARIZADO,95
4,2022-03-31,BANCARIZADO,15475
5,2022-03-31,NO BANCARIZADO,155
6,2022-06-30,BANCARIZADO,16695
7,2022-06-30,NO BANCARIZADO,167
8,2022-09-30,BANCARIZADO,17937
9,2022-09-30,NO BANCARIZADO,193


## 4. Cruce Bancarizado × Desempeño

Vista cruzada para dimensionar cuántos clientes quedarán fuera del modelado por cada
motivo de exclusión.

In [24]:
# %% Cruce: bancarización vs desempeño
pd.crosstab(info["MARCA_BANCARIZADO"], info["SIN_DESEMPENO"])

SIN_DESEMPENO,CON_DESEMPENO,SIN_DESEMPENO
MARCA_BANCARIZADO,,
BANCARIZADO,77909,950
NO BANCARIZADO,703,29


## 5. Mora severa en el primer mes

`MARCA_VENCIDO` marca los créditos que ya llegan **muy deteriorados** al punto de
observación (más de 180 días de mora en M1). Son casos atípicos que se separan del
análisis.

In [25]:
# %% MARCA_VENCIDO: mora en el primer mes (M1) mayor a 180 días
info["MARCA_VENCIDO"] = np.where(info["NUMERO_DIAS_MOROSIDAD_OP_M1"] > 180, 1, 0)

info.groupby("MARCA_VENCIDO").size().reset_index(name="N")

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/3539450838.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["MARCA_VENCIDO"] = np.where(info["NUMERO_DIAS_MOROSIDAD_OP_M1"] > 180, 1, 0)


,MARCA_VENCIDO,N
0,0,62038
1,1,17553


## 6. Máximo de mora en la ventana de desempeño

`MAX_DIAS_MOROSIDAD` = la **peor** morosidad alcanzada en cualquiera de los meses
M2–M13. Es el indicador central para decidir bueno / malo.

In [26]:
# %% MAX_DIAS_MOROSIDAD: peor mora observada en la ventana de desempeño (M2..M13)
cols_mora = [f"NUMERO_DIAS_MOROSIDAD_OP_M{i}" for i in range(2, 14)]
info["MAX_DIAS_MOROSIDAD"] = info[cols_mora].max(axis=1)

info["MAX_DIAS_MOROSIDAD"].describe()

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/1444430080.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["MAX_DIAS_MOROSIDAD"] = info[cols_mora].max(axis=1)


count    79591.000000
mean       408.629028
std        924.467091
min          0.000000
25%          0.000000
50%         31.000000
75%        360.000000
max      13809.000000
Name: MAX_DIAS_MOROSIDAD, dtype: float64

## 7. Definición de la Variable Dependiente


| Orden | Condición | VarDep |
|---|---|---|
| 1 | No bancarizado | **5** |
| 2 | Sin desempeño (< 6 meses con saldo) | **4** |
| 3 | Mora en M1 > 180 días | **3** |
| 4 | Máx. mora M2–M13 > 60 días | **1 · malo** |
| 5 | Máx. mora M2–M13 = 0 | **0 · bueno** |
| — | Cualquier otro caso (mora 1–60) | **2 · indeterminado** |

El modelo se entrenará solo con **0** y **1**.

In [27]:
# %% Definición de la VarDep 
conditions = [
    info["MARCA_BANCARIZADO"] == "NO BANCARIZADO",   # -> 5
    info["SIN_DESEMPENO"] == "SIN_DESEMPENO",        # -> 4
    info["NUMERO_DIAS_MOROSIDAD_OP_M1"] > 180,       # -> 3
    info["MAX_DIAS_MOROSIDAD"] > 60,                 # -> 1  (malo)
    info["MAX_DIAS_MOROSIDAD"] == 0,                 # -> 0  (bueno)
]
choices = [5, 4, 3, 1, 0]
info["VarDep"] = np.select(conditions, choices, default=2)   # 2 = indeterminado

info.groupby("VarDep").size().reset_index(name="N").sort_values("VarDep")

/var/folders/v6/d_4300n96432y2g47c2hlw640000gn/T/ipykernel_2058/1346202959.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  info["VarDep"] = np.select(conditions, choices, default=2)   # 2 = indeterminado


,VarDep,N
0,0,24507
1,1,16886
2,2,19054
3,3,17462
4,4,950
5,5,732


## 8. Eliminar las variables de la ventana de desempeño

Las columnas `*_OP_M*` describen el **futuro** del crédito (los 13 meses posteriores
a la observación) y se usaron para construir la `VarDep`.

**No pueden ser variables predictoras:** usarlas sería *fuga de datos* — el modelo
"vería" la respuesta. Por eso se eliminan; las predictoras son solo el historial de
buró construido en el notebook `01`.

In [ ]:
# Quitar las columnas de la ventana de desempeño
cols_op_m = [c for c in info.columns if "OP_M" in c]
info = info.drop(columns=cols_op_m)

print("Columnas de desempeño eliminadas:", len(cols_op_m))
print("Dimensiones finales:", info.shape)

Columnas de desempeño eliminadas: 65
Dimensiones finales: (79591, 2506)


## 9. Guardar la base de modelamiento

Se guarda `InfoModelamiento.pkl`: la base con la `VarDep` definida y sin las columnas
de desempeño — lista para `03_Modelamiento.ipynb`.

In [ ]:
info.to_pickle("data/buro/InfoModelamiento.pkl")
print("Guardado en BDD/InfoModelamiento.pkl")

Guardado en BDD/InfoModelamiento.pkl
